# Допуск: NoLiMa: Long-Context Evaluation Beyond Literal Matching

**Статья:** A. Modarressi, H. Deilamsalehy, F. Dernoncourt, T. Bui, R. Rossi, S. Yoon,
H. Schütze, 2025 — <https://arxiv.org/abs/2502.05167>

5 заданий. Под заголовком каждого задания указан его тип:

- **что есть что** — названия функций заменены: определите, что чему соответствует, и
  ответьте на вопросы;
- **расхождение со статьёй** — код реализует не то, что описано в статье: найдите это место;
- **неверный комментарий** — код корректен, но один комментарий утверждает неправду: найдите
  его;
- **вопрос по коду** — в коде ошибок нет, ответьте на вопросы по статье, опираясь на код.

Для найденного отклонения напишите: где оно, что не так, какое место статьи (раздел /
формула / таблица / приложение / цитата) нарушено и как должно быть. Код писать не нужно —
достаточно объяснить словами. Запускать и отлаживать код тоже не нужно: все задания решаются
чтением кода и статьи.

Общие импорты:

In [ ]:
import random
import torch

`tok` — токенизатор instruction-модели из Hugging Face, `llm(prompt) -> str` — вызов модели
с жадным декодированием. Сниппеты — части одной реализации; функции, которые используются,
но не показаны, считайте корректными.

## Задание 1. Пять пар «игла — вопрос»

**Тип задания:** что есть что — названия функций заменены, определите, что чему соответствует.

Каждая функция возвращает пару (игла, вопрос). Определите, какая функция даёт:
основной тест NoLiMa с одним скрытым шагом (one-hop, порядок по умолчанию); two-hop;
вариант с обратным порядком (inverted); вариант **Direct** и вариант **Multiple Choice**
из абляции §4.4.4. Ошибок в коде нет.

In [ ]:
fact = {"char": "Yuki", "w_n": "the Semper Opera House",
        "w_q": "Dresden", "w_q2": "the state of Saxony"}

def make_a(f):
    return (f"Actually, {f['char']} lives next to {f['w_n']}.",
            f"Which character has been to {f['w_q']}?")

def make_b(f):
    return (f"Actually, {f['char']} lives next to {f['w_n']}.",
            f"Which character has been to {f['w_q']}? Alex, Sarah, {f['char']}, or Rebecca?")

def make_c(f):
    return (f"{f['w_n'][0].upper() + f['w_n'][1:]} is next to where {f['char']} lives.",
            f"Which character has been to {f['w_q']}?")

def make_d(f):
    return (f"Actually, {f['char']} lives next to {f['w_n']}.",
            f"Which character lives next to {f['w_n']}?")

def make_e(f):
    return (f"Actually, {f['char']} lives next to {f['w_n']}.",
            f"Which character has been to {f['w_q2']}?")

**Вопросы.**
1. Посчитайте ROUGE-1 precision вопроса относительно иглы для `make_a` и `make_d` (доля
   слов вопроса, которые есть в игле). Почему в табл. 1 авторы берут именно precision, а не
   recall (сноска 2), и что означает 0.069 у NoLiMa?
2. В `make_b` вопрос про ту же связь Dresden → Semper Opera House, что и в `make_a`, но по
   табл. 6 точность на 32K растёт с 56.2 до 93.1. Где здесь появляется literal match, если
   слова `w_n` в вопросе нет?
3. Какая из функций `make_a` и `make_c` даёт более трудный тест (рис. 2b)? Как авторы
   объясняют это через причинное (causal) внимание (§4.4.1)?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 2. Фильтрация стога

**Тип задания:** неверный комментарий — код корректен, но один комментарий утверждает неправду.

Код удаляет из книжного текста места, которые могут оказаться непреднамеренным ответом на
вопросы из набора игл (рис. 1). `prompt` — инструкция и 4 примера из рис. 1, `review` —
функция, которая по найденным кандидатам возвращает фрагменты для удаления. Найдите одно
отклонение от статьи.

In [ ]:
def chunks(text, size=1000, stride=800):           # characters, ≈ 250 tokens
    # short chunks only save compute: run on the whole haystack at once,
    # the filter model would flag exactly the same spans
    return [text[i:i + size] for i in range(0, len(text), stride)]

def scan(text, questions, llm, prompt):
    flagged = []
    for chunk in chunks(text):
        for q in questions:
            reply = llm(prompt.format(chunk=chunk, question=q))
            if "N/A" not in reply:
                flagged.append((chunk, q, reply))   # a possible unintended answer
    return flagged

def filter_haystack(text, questions, llm, prompt, review):
    # review: one of the authors reads the flagged chunks and picks the spans to delete
    while True:
        spans = review(scan(text, questions, llm, prompt))
        if not spans:
            return text                            # repeat until no further removal
        for s in spans:
            text = text.replace(s, "")

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 3. Куда ставить иглу

**Тип задания:** вопрос по коду — ошибок нет, ответьте на вопросы.

Две функции задают позиции иглы (в токенах) для одной длины контекста; `make_input` вставляет
иглу в стог и собирает запрос к модели по шаблону из прил. C.

In [ ]:
def full_sweep(ctx_len, n=26):
    return [round(i * ctx_len / (n - 1)) for i in range(n)]

def last_2k(ctx_len, n=26, span=2000):
    return [ctx_len - span + round(i * span / (n - 1)) for i in range(n)]

def make_input(tok, hay_ids, needle, question, pos, template):
    # hay_ids: haystack cut to ctx_len tokens; the question comes after the haystack
    needle_ids = tok("\n" + needle + "\n", add_special_tokens=False).input_ids
    ids = hay_ids[:pos] + needle_ids + hay_ids[pos:]
    text = template.format(haystack=tok.decode(ids), question=question)
    return tok.apply_chat_template([{"role": "user", "content": text}],
                                   tokenize=False, add_generation_prompt=True)

**Вопросы.**
1. Пусть `ctx_len` = 4000 и 32000, `i = 13`. На каком расстоянии (в токенах стога) от
   вопроса оказывается игла в `full_sweep` и в `last_2k`? Что меняется при переходе от 4K к
   32K в каждом случае (рис. 4)?
2. На рис. 3d (two-hop, last 2K) кривые для разных длин почти плоские по глубине, но лежат
   тем ниже, чем длиннее контекст. Почему авторы делают из этого вывод, что падение не
   объясняется позиционным кодированием (RoPE), и что тогда остаётся причиной (§4.4.2)?
3. Зачем игла обёрнута в переводы строк, а все шаблоны игл (табл. 7) начинаются с вводной
   фразы вроде «Actually,» или «In 2013,» (прил. A)?

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 4. Base score и эффективная длина

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

`acc[e, h, l]` — точность на примере «игла — вопрос» `e` в стоге `h` при длине `LENGTHS[l]`
(доля из 26 позиций иглы, где ответ содержит верное имя). Найдите одно отклонение от §4.3.

In [ ]:
LENGTHS = [250, 500, 1000, 2000, 4000, 8000, 16000, 32000]

def base_score(acc):
    # acc: (E, H, L); E = 58 question-needle pairs, H = 5 haystacks
    per_len = acc.mean(1)                            # (E, L) average over haystacks
    best_short = per_len[:, :3].max(1).values        # (E,) best of 250 / 500 / 1K
    return 100 * best_short.mean().item()

def scores_by_length(acc):
    return {L: 100 * acc[:, :, i].mean().item() for i, L in enumerate(LENGTHS)}

def effective_length(scores, base):
    threshold = 85.6                                 # Llama 2 at 4K, as in RULER
    ok = [L for L in LENGTHS[2:] if scores[L] > threshold]
    return max(ok) if ok else "<1K"

def normalized(scores, base):
    return {L: s / base for L, s in scores.items()}  # y-axis of figs. 2 and 5

**Ваш ответ:**

_Напишите ответ здесь._

## Задание 5. Отвлекающее предложение

**Тип задания:** расхождение со статьёй — код реализует не то, что описано в статье.

`hay_ids` — стог (список id токенов), в который игла уже вставлена с позиции `needle_pos`
и занимает `needle_len` токенов; `f` — словарь как в задании 1. Найдите одно отклонение от
абляции с дистрактором (§4.4.4, прил. D).

In [ ]:
def add_distractor(tok, hay_ids, f, needle_pos, needle_len):
    topic = f["w_n"]                                   # what the distractor sentence mentions
    sent = tok(f"\nThere was an article about {topic} in the daily newspaper.\n",
               add_special_tokens=False).input_ids
    n = len(hay_ids)
    allowed = [p for p in range(int(0.2 * n), int(0.8 * n))               # 20%–80% of the context
               if p <= needle_pos - 0.2 * n                                # >= 20% before the needle
               or p >= needle_pos + needle_len + 0.2 * n]                  # or >= 20% after it
    p = random.choice(allowed)
    return hay_ids[:p] + sent + hay_ids[p:]

**Ваш ответ:**

_Напишите ответ здесь._